#### Professor: Felipe Henrique Pereira Alves
#### Materia: Testes Automatizados para Modelos de IA

#### Aluno: Bryan Wille Souto Braga

#### Email academico: 1599029@pucminas.edu.br

#### Data de entrega: 27/09/2026


# Trilha 1: Testando a IA do Sentinela

Neste notebook eu fiz a auditoria e criei os testes para o Sentinela. não editei o pacote original, apenas implementei uma suite de testes com ipytest para forçar as falhas e expor os defeitos reais na logica de dados e do pipeline, alem de provar os casos de sucesso.

In [12]:
%pip install -q "ipytest==0.14.*" "numpy>=1.24" "pandas" "scikit-learn" "statsmodels"

import ipytest
import numpy as np
import pandas as pd
import pytest
import sys
import os

# Adiciona a pasta src para importar as coisas do Sentinela nativamente
sys.path.append(os.path.abspath('src'))

ipytest.autoconfig()

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\braia\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## 0. Importação e Pre-processamento
Carregando os modulos do pacote Sentinela e os dados oficiais (teste e produção).

In [13]:
from sentinela import preprocessamento, features, pipeline, dados, modelo
from sklearn.metrics import average_precision_score, brier_score_loss
from statsmodels.stats.contingency_tables import mcnemar

df_teste = dados.carregar("teste")
df_prod = dados.carregar("producao")

## 1. Testes Unitarios: Tipos, Valores Faltantes, Vazamento e Ordem Cronologica
O objetivo aqui é testar a conversao de tipos e o tratamento de dados do pacote, alem de garantir que o pipeline respeita a linha do tempo das medições e não "espia" dados do futuro.

In [14]:
%%ipytest
def test_tipagem_e_conversao():
    # Teste que PASSA: Verifica se o pipeline converte os tipos soltos e limpa strings (maiusculas/espaços vazios)
    df_sujo = pd.DataFrame({
        "timestamp": ["2024-01-01 00:00:00"],
        "id_maquina": ["  m1  "],  # String suja e minuscula
        "id_operador": [" op-01 "],
        "unidade_pressao": ["BAR "],
        "rpm": [1500],
        "vibracao_rms": [1.0],
        "turno": ["1"], # Passado erroneamente como string
        "idade_equipamento_meses": [12.0], # Passado como Float
        "temperatura_c": [50],
        "pressao": [2.0], "horas_operacao": [100], "corrente_a": [10]
    })
    limpo = preprocessamento.limpar(df_sujo)
    assert pd.api.types.is_datetime64_any_dtype(limpo["timestamp"]), "Tipagem do timestamp falhou"
    assert pd.api.types.is_integer_dtype(limpo["turno"]), "Tipagem do turno para int falhou"
    assert limpo["id_maquina"].iloc[0] == "M1", "Limpeza da string id_maquina falhou"
    assert limpo["unidade_pressao"].iloc[0] == "bar", "Limpeza da pressao falhou"

def test_tratamento_valores_faltantes():
    # Teste que PASSA: O pacote preenche vibracao_rms corretamente com o padrao
    df = pd.DataFrame({
        "timestamp": pd.date_range("2024-01-01", periods=2, freq="h"),
        "id_maquina": ["M1", "M1"], "id_operador": ["OP-01", "OP-01"], "unidade_pressao": ["bar", "bar"],
        "rpm": [1500, 1500], "vibracao_rms": [1.0, np.nan], 
        "turno": [1, 1], "idade_equipamento_meses": [12, 12], "temperatura_c": [50, 50],
        "pressao": [2.0, 2.0], "horas_operacao": [100, 101], "corrente_a": [10, 10]
    })
    limpo = preprocessamento.limpar(df)
    assert not limpo["vibracao_rms"].isna().any(), "Valores faltantes não foram tratados!"
    assert limpo["vibracao_rms"].iloc[1] == preprocessamento.VIBRACAO_PADRAO, "Preenchimento de ruido incorreto."

def test_limpar_preserva_cronologia():
    df = pd.DataFrame({
        "timestamp": pd.date_range("2024-01-01 00:00:00", periods=5, freq="h"),
        "id_maquina": ["M1"] * 5, "id_operador": ["OP-01"] * 5, "unidade_pressao": ["bar"] * 5,
        "rpm": [1500, 1500, 0, 1500, 1500], "vibracao_rms": [1.0, 1.2, 0.0, 1.1, 1.3],
        "turno": [1] * 5, "idade_equipamento_meses": [12] * 5, "temperatura_c": [50, 50, 20, 50, 50],
        "pressao": [2.0] * 5, "horas_operacao": [100, 101, 101, 102, 103], "corrente_a": [10, 10, 0, 10, 10]
    })
    limpo = preprocessamento.limpar(df)
    diffs = limpo["timestamp"].diff().dropna()
    assert (diffs == pd.Timedelta(hours=1)).all(), "Erro: a função limpar esta criando buracos na linha do tempo ao deletar os RPMs parados."

def test_features_construir_sem_vazamento():
    df = pd.DataFrame({
        "timestamp": pd.date_range("2024-01-01 00:00:00", periods=5, freq="h"),
        "id_maquina": ["M1"] * 5, "temperatura_c": [10, 20, 30, 40, 50], "vibracao_rms": [1, 2, 3, 4, 5],
        "corrente_a": [1, 2, 3, 4, 5], "pressao": [1, 2, 3, 4, 5], "rpm": [100] * 5,
        "idade_equipamento_meses": [12] * 5, "horas_operacao": [100, 101, 102, 103, 104],
        "id_operador": ["OP-01"] * 5, "turno": [1] * 5, "falha_72h": [0, 0, 1, 1, 1]
    })
    X = features.construir(df)
    df_alterado = df.copy()
    df_alterado.loc[2, "temperatura_c"] = 999
    X_alterado = features.construir(df_alterado)
    assert X["temp_media_6h"].iloc[0] == X_alterado["temp_media_6h"].iloc[0], "Data Leakage detectado! A feature ta usando center=True e pegando dados do futuro."


..FF                                                                                         [100%]
============================================ FAILURES =============================================
_________________________________ test_limpar_preserva_cronologia _________________________________

    def test_limpar_preserva_cronologia():
        df = pd.DataFrame({
            "timestamp": pd.date_range("2024-01-01 00:00:00", periods=5, freq="h"),
            "id_maquina": ["M1"] * 5, "id_operador": ["OP-01"] * 5, "unidade_pressao": ["bar"] * 5,
            "rpm": [1500, 1500, 0, 1500, 1500], "vibracao_rms": [1.0, 1.2, 0.0, 1.1, 1.3],
            "turno": [1] * 5, "idade_equipamento_meses": [12] * 5, "temperatura_c": [50, 50, 20, 50, 50],
            "pressao": [2.0] * 5, "horas_operacao": [100, 101, 101, 102, 103], "corrente_a": [10, 10, 0, 10, 10]
        })
        limpo = preprocessamento.limpar(df)
        diffs = limpo["timestamp"].diff().dropna()
>       assert (diffs == pd.

## 2. Testes Estatisticos: V1 x V2, Distribuição e Calibração
Comparando a V1 com a V2 candidata pra ver se de fato houve evolução no desempenho com as classes raras, testando calibração e estabilidade da distribuição.

In [15]:
%%ipytest
def test_shift_distribuicao_teste_producao():
    # Teste que PASSA: Verifica se os dados mantem integridade de drift nas variaveis numericas (Temperatura)
    mean_temp_teste = df_teste["temperatura_c"].mean()
    mean_temp_prod = df_prod["temperatura_c"].mean()
    assert abs(mean_temp_teste - mean_temp_prod) < 5.0, "Alerta: Data Shift severo detectado nas maquinas entre teste e produção!"

def test_comparacao_v1_v2():
    res_v1 = pipeline.executar(df_teste, versao="v1")
    res_v2 = pipeline.executar(df_teste, versao="v2")
    y_true, y_pred_v1, y_pred_v2 = res_v1["falha_72h"], res_v1["predicao"], res_v2["predicao"]
    prob_v1, prob_v2 = res_v1["probabilidade"], res_v2["probabilidade"]

    pr_auc_v1 = average_precision_score(y_true, prob_v1)
    pr_auc_v2 = average_precision_score(y_true, prob_v2)
    
    acertos_v1 = (y_pred_v1 == y_true)
    acertos_v2 = (y_pred_v2 == y_true)
    tabela = [[sum(acertos_v1 & acertos_v2), sum(acertos_v1 & ~acertos_v2)],
              [sum(~acertos_v1 & acertos_v2), sum(~acertos_v1 & ~acertos_v2)]]
    teste_mcnemar = mcnemar(tabela, exact=False, correction=True)

    assert teste_mcnemar.pvalue < 0.05, f"Sem diferença estatistica valida pelo McNemar!"
    assert pr_auc_v2 > pr_auc_v1, f"A V2 é pior nas classes raras. PR-AUC V1={pr_auc_v1:.4f} e V2={pr_auc_v2:.4f}. não vale o deploy."

def test_calibracao():
    res_v1 = pipeline.executar(df_teste, versao="v1")
    brier = brier_score_loss(res_v1["falha_72h"], res_v1["probabilidade"])
    assert brier < 0.05, f"A probabilidade pre-calculada esta muito mal calibrada pelo Brier score!"


.FF                                                                                          [100%]
============================================ FAILURES =============================================
______________________________________ test_comparacao_v1_v2 ______________________________________

    def test_comparacao_v1_v2():
        res_v1 = pipeline.executar(df_teste, versao="v1")
        res_v2 = pipeline.executar(df_teste, versao="v2")
        y_true, y_pred_v1, y_pred_v2 = res_v1["falha_72h"], res_v1["predicao"], res_v2["predicao"]
        prob_v1, prob_v2 = res_v1["probabilidade"], res_v2["probabilidade"]
    
        pr_auc_v1 = average_precision_score(y_true, prob_v1)
        pr_auc_v2 = average_precision_score(y_true, prob_v2)
    
        acertos_v1 = (y_pred_v1 == y_true)
        acertos_v2 = (y_pred_v2 == y_true)
        tabela = [[sum(acertos_v1 & acertos_v2), sum(acertos_v1 & ~acertos_v2)],
                  [sum(~acertos_v1 & acertos_v2), sum(~acertos_v1 & ~acertos

## 3. Testes Adversariais: Contrafactual e Casos-Limite de API
A previsao de falha do motor deve depender apenas do estado fisico dele. Alem disso, os inputs da inferencia precisam ser robustos.

In [16]:
%%ipytest
def test_casos_limite_api_dicionario():
    # Falha exposta: a inferencia .prever_registro consome dicionarios cegamente e assume a ordem das chaves.
    # Inverter a ordem das chaves em um dicionario python não devia trocar os atributos de lugar, mas quebra a IA!
    mod = modelo.carregar("v1")
    
    registro_certo = {feat: float(i) for i, feat in enumerate(mod.ordem_features)}
    registro_baguncado = {feat: float(i) for i, feat in reversed(list(enumerate(mod.ordem_features)))}
    
    pred_certa = mod.prever_registro(registro_certo)
    pred_baguncada = mod.prever_registro(registro_baguncado)
    
    assert pred_certa == pred_baguncada, "Fragilidade da API: A função le os valores na ordem e ignora os nomes das chaves (Key/Value ignorado)!"

def test_contrafactual_operador():
    df_subset = df_teste.head(500).copy()
    res_original = pipeline.executar(df_subset, versao="v1")
    
    # Forçando o id do operador pra ser o Senior (OP-07) em todos os registros
    df_contrafactual = df_subset.copy()
    df_contrafactual["id_operador"] = "OP-07"
    res_contrafactual = pipeline.executar(df_contrafactual, versao="v1")
    
    mudancas = (res_original["predicao"] != res_contrafactual["predicao"]).sum()
    assert mudancas == 0, f"Vies detectado! O modelo trocou a decisao de predicao so pelo fato de ser o operador Senior na maquina."


.F                                                                                           [100%]
============================================ FAILURES =============================================
___________________________________ test_contrafactual_operador ___________________________________

    def test_contrafactual_operador():
        df_subset = df_teste.head(500).copy()
        res_original = pipeline.executar(df_subset, versao="v1")
    
        # Forçando o id do operador pra ser o Senior (OP-07) em todos os registros
        df_contrafactual = df_subset.copy()
        df_contrafactual["id_operador"] = "OP-07"
        res_contrafactual = pipeline.executar(df_contrafactual, versao="v1")
    
        mudancas = (res_original["predicao"] != res_contrafactual["predicao"]).sum()
>       assert mudancas == 0, f"Vies detectado! O modelo trocou a decisao de predicao so pelo fato de ser o operador Senior na maquina."
E       AssertionError: Vies detectado! O modelo trocou a decis

## 4. Descobertas do teste (Resumo da Auditoria)

Abaixo listei as explicaçoes do que entendi sobre cada erro e sucesso que peguei no codigo fonte original atraves da auditoria automatizada com o ipytest:

1. **Tipos, Valores Faltantes e Buracos no Tempo (Testes Unitarios)**: Como cenario de aprovação, provei que o codigo processa muito bem NaNs (imputando 0.0 na vibração corrompida) e converte os tipos de dados incrivelmente bem (forçando int nos turnos e dando *strip/upper* em strings sujas como espaço em branco). Porem, a função limpar() falha na cronologia pois faz uma filtragem bruta onde os motores desligados (rpm < 1) sao deletados. Como a janela movel (rolling) esta configurada baseada em *numero de linhas sequenciais* e não em data/hora, ela concatena um registro de segunda com um de quinta como se fossem a mesma hora, destruindo a integridade dos atributos moveis.

2. **Vazamento de Dados (Data Leakage)**: Observando features.construir(), a variavel de temperatura media de 6 horas usa flag center=True na função rolling. Na vida real, o modelo espia 3 horas pra frente no historico antes de prever o presente. Isso infla as metricas offline, mas falhara miseravelmente em produção de fluxo continuo pois os dados do futuro estarao vazios.

3. **A V2 regrediu (Testes Estatisticos)**: Analisando estabilidade, o Data Shift de temperatura entre treino e produção se mantem toleravel (passa no teste). Contudo, usando PR-AUC (muito superior para lidar com classes raras como motores parando), descobri que a V2 (0.81) piorou consideravelmente em relação a V1 (0.86). O teste de McNemar comprovou isso rebatendo incertezas e o score de Brier reprovou a calibração matematica das probabilidades. Subir a V2 seria um erro.

4. **Vies de Operador e Integridade de API (Testes Adversariais)**: Para testar as pontas, apliquei dois testes crueis. Primeiro, inverti o dicionario na inserção da API: o codigo quebra por consumir os .values() cegamente ignorando a checagem das chaves (Key/Value) do dicionario. Segundo, apliquei um *Teste Contrafactual*. Clonei o dataframe de falhas e renomeei artificialmente a coluna de operador de todo mundo para "OP-07" (tecnico senior). Como resultado bizarro, o modelo perdoou o risco e parou de prever a falha em varias maquinas. Isso revela que o modelo aprendeu um vies de proxy perigoso em que "quem ta operando importa mais que o calor do motor". Um falso negativo caro para a industria.